# Setup

In [ ]:
packages <- c("osfr", "tidyverse", "jsonlite")

install.packages(packages)

In [ ]:
# OSF auth token is read from the environment -- never hardcode it.
# In Colab, store it as a secret and expose it to the R kernel before running, e.g.
#   Sys.setenv(OSF_TOKEN = "<your token>")   # in a cell you do NOT commit
osf_access_token <- Sys.getenv("OSF_TOKEN")
stopifnot("OSF_TOKEN env var is not set" = nzchar(osf_access_token))

private_json_osf_component <- "vmdu6"
private_jpg_osf_component <- "3b6et"
private_webm_osf_component <- "34t8d"
private_osf_overall_component <- "2wcgp"

public_jpg_osf_component <- "7s2ae"
public_json_osf_component <- "6cn7a"
#######################

library(osfr)
library(tidyverse)
library(jsonlite)

In [ ]:
dir.create("data")

# Download data from OSF

In [ ]:
osf_auth(osf_access_token)

a <- osf_retrieve_node(private_json_osf_component) %>%
  osf_ls_files()  %>%
  osf_download(path = "data", conflicts = "skip", progress = TRUE)

b <- osf_retrieve_node(private_jpg_osf_component) %>%
  osf_ls_nodes() %>% pull(id)

for(node in b){
  xx <- osf_retrieve_node(node) %>%
    osf_ls_files() %>%
    osf_download(path = "data", conflicts = "skip", progress = TRUE)
}


In [ ]:
# List the part files
part_files <- list.files('data', pattern = "jpg.tar.gz.part.*", full.names = T)

# Sort the files to ensure they're in the correct order
part_files <- sort(part_files)

# Create the command to concatenate the files
cmd <- paste("cat", paste(part_files, collapse=" "), "> data/jpg.tar.gz")

# Run the command
system(cmd)

In [ ]:
paste("cat", paste(part_files, collapse=" "), "> data/jpg.tar.gz")

In [ ]:
untar("data/json.tar.gz", exdir = "data/json")
untar("data/jpg.tar.gz", exdir = "data/jpg")
#system("tar -xzf data/jpg.tar.gz -C data/jpg")

In [ ]:
file.remove("data/jpg.tar.gz")
file.remove("data/json.tar.gz")

# Gather prolific JSON data

In [ ]:
json_data <- list.files(
  path = "data/json",
  pattern = "^\\w{8}.json$",
  full.names = TRUE
) %>%
  lapply(fromJSON) %>%
  bind_rows()

In [ ]:
prolific_info <- json_data %>%
  select(subject_id, prolific_id) %>%
  group_by(subject_id) %>%
  slice_head(n = 1) %>%
  arrange(prolific_id)

In [ ]:
data_trials <- c(
  "webcam-type",
  "glasses",
  "browser-check",
  "public-videos",
  "research-videos"
)

experiment_data <- json_data %>%
  filter((task %in% data_trials) | (trial_type == "html-video-response"))

# Filter subjects

In [ ]:
# people who contacted us asking to remove their videos
# Prolific IDs of participants who asked to be removed, one per line. Kept out of the repository.
excluded_file <- "excluded_participants.txt"
manually_excluded_participants <- if (file.exists(excluded_file)) readLines(excluded_file) else character(0)
print(paste('There are',length(manually_excluded_participants),'participants who requested to be remove from the dataset'))

In [ ]:
total_participants <- experiment_data %>% pull(subject_id) %>% unique() %>% length()

people_who_give_permission_for_videos <- experiment_data %>%
  filter(task == "public-videos") %>%
  select(subject_id, response) %>%
  filter(response == 0) %>%
  filter(!subject_id %in% manually_excluded_participants) %>%
  pull(subject_id)

print(paste('There are',length(people_who_give_permission_for_videos),'people who consented to public release of their videos'))
print(paste(total_participants, 'completed the experiment'))

In [ ]:
jpg_files <- list.files(
  path = "data/jpg",
  pattern = ".jpg$",
)

jpg_files_per_subject <- data.frame(file_name = jpg_files) %>%
  mutate(subject_id = substring(file_name, 1, 8)) %>%
  group_by(subject_id) %>%
  count()

subjects_with_complete_data <- jpg_files_per_subject %>%
  filter(n == 144) %>%
  pull(subject_id)

subjects_with_complete_data_and_permission <- subjects_with_complete_data %>%
  intersect(people_who_give_permission_for_videos)

print(paste('There are',length(subjects_with_complete_data),'who generated a complete dataset'))
print(paste('There are',length(subjects_with_complete_data_and_permission),'who gave permission and have a complete dataset'))

# Describe dataset characteristics

In [ ]:
filtered_dataset <- experiment_data %>%
  filter(subject_id %in% subjects_with_complete_data_and_permission)

## Webcam type

In [ ]:
webcam_info <- filtered_dataset %>%
  filter(task == "webcam-type") %>%
  select(subject_id, response) %>%
  mutate(
    webcam_type = factor(
      response,
      labels = c("integrated", "external"),
      levels = c(0, 1)
    )
  ) %>%
  select(-response)

webcam_info %>% count(webcam_type)

## Wearing glasses

In [ ]:
glasses_info <- filtered_dataset %>%
  filter(task == "glasses") %>%
  select(subject_id, response) %>%
  mutate(
    wore_glasses = factor(
      response,
      labels = c("no", "yes"),
      levels = c(1, 0)
    )
  ) %>%
  select(-response)

glasses_info %>% count(wore_glasses)

In [ ]:
write.csv(glasses_info, file = "/content/data/glasses_info.csv", row.names = FALSE)
glasses_node <- osf_retrieve_node('yezk8') %>%
  osf_upload("data/glasses_info.csv", conflicts="overwrite")

## Screen characteristics

In [ ]:
screen_info <- experiment_data %>%
  filter(task == "browser-check") %>%
  select(subject_id, aspect_ratio, width, height)


In [ ]:
screen_info %>% count(aspect_ratio, width, height) %>% arrange(desc(n)) %>% slice_head(n=20)

## Dot detection task

In [ ]:
rt_data <- json_data %>%
  filter(task == "dot_detection") %>%
  select(rt, subject_id)

In [ ]:
detection_task_data <- rt_data %>%
  group_by(subject_id) %>%
  summarize(
    mean = mean(rt), median = median(rt), sd = sd(rt),
    "prop<200ms" = sum(rt < 200) / n(),
    "prop<400ms" = sum(rt < 400) / n(),
    "prop<800ms" = sum(rt < 800) / n(),
    "prop<1600ms" = sum(rt < 1600) / n(),
    "prop<3200ms" = sum(rt < 3200) / n()
  )



In [ ]:
ggplot(rt_data, aes(x=rt, group=subject_id)) +
  coord_cartesian(xlim=c(0,3200))+
  stat_ecdf()

In [ ]:
detection_task_data %>% slice_head(n=10)

## Interaction data

In [ ]:
interaction_data <- list.files(
  path = "data/json",
  pattern = "_interactions.json$",
  full.names = TRUE
) %>%
lapply(function(x){
  d <- fromJSON(x) %>%
  mutate(subject_id = substring(x, 11, 18))
  return(d)
}) %>%
bind_rows()

interaction_data_wide <- interaction_data %>%
  group_by(subject_id) %>%
  filter(trial >= 9 & trial <= 340) %>%
  count(event) %>%
  pivot_wider(id_cols = subject_id, names_from = event, values_from = n)

interaction_data_wide[is.na(interaction_data_wide)] <- 0

### Did anyone exit fullscreen before the final trial?

In [ ]:
fullscreenexit_trial <- interaction_data %>%
  filter(event == "fullscreenexit") %>%
  group_by(subject_id) %>%
  slice_head(n = 1) %>%
  select(subject_id, trial) %>%
  rename("fullscreen_exit_trial" = "trial")

fullscreenenter_trial <- interaction_data %>%
  filter(event == "fullscreenenter") %>%
  group_by(subject_id) %>%
  slice_tail(n = 1) %>%
  select(subject_id, trial) %>%
  rename("fullscreen_enter_trial" = "trial")

fullscreen_data <- fullscreenenter_trial %>% left_join(fullscreenexit_trial, by=join_by(subject_id))

In [ ]:
suspicious_fullscreen_behavior <- fullscreen_data %>% filter(fullscreen_exit_trial < 339)

In [ ]:
print(paste(nrow(suspicious_fullscreen_behavior), 'subjects may have exited fullscreen mode before completing the experiment'))

# Upload filtered dataset to OSF

In [ ]:
### create filtered jpg.tar.gz

public_jpg_files <- jpg_files[str_extract(jpg_files, "^[^_]+")  %in% subjects_with_complete_data_and_permission]

tar(
  tarfile = "data/jpg.tar.gz",
  files = file.path("data/jpg", public_jpg_files),
  compression = "gzip"
)

a <- osf_retrieve_node(public_jpg_osf_component) %>%
  osf_upload("data/jpg.tar.gz", conflicts="overwrite")

In [ ]:
writeLines(toJSON(
  subjects_with_complete_data_and_permission
), "data/subjects-with-permission.json")

a <- osf_retrieve_node(private_osf_overall_component) %>%
  osf_upload("data/subjects-with-permission.json", conflicts="overwrite")


In [ ]:
file.info("data/jpg.tar.gz")$size / 1000000